# 第24章：调试与性能分析

## 前置知识
- 完成 Part 1-3

## 学习目标
- 掌握 Triton kernel 的调试方法
- 学会使用 `TRITON_INTERPRET=1` 解释器模式
- 理解常见错误和解决方法
- 学会性能分析和瓶颈定位

In [ ]:
import torch
import triton
import triton.language as tl
import os

## 24.1 TRITON_INTERPRET 模式

Triton 提供了解释器模式，可以在 CPU 上逐步执行 kernel：

```bash
# 方式1: 环境变量
TRITON_INTERPRET=1 python my_script.py

# 方式2: 在代码中设置
os.environ['TRITON_INTERPRET'] = '1'
```

### 解释器模式的好处
- 可以用 `print()` 调试（GPU kernel 不能用 print）
- 可以设断点
- 不需要 GPU
- 所有操作在 CPU 上用 NumPy 执行

### 限制
- 性能分析无意义（CPU 执行）
- 某些 GPU 特有行为可能不同
- `tl.dot` 精度可能与 GPU Tensor Core 不同

## 24.2 tl.device_print — GPU 上的 print

Triton 提供了 `tl.device_print` 在 GPU 上打印调试信息：

In [ ]:
@triton.jit
def debug_kernel(x_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask)
    
    # 只在 pid=0 时打印，避免输出爆炸
    if pid == 0:
        tl.device_print("pid", pid)
        tl.device_print("first element", tl.load(x_ptr))
    
    result = x * 2.0
    tl.store(out_ptr + offsets, result, mask=mask)

x = torch.tensor([1.0, 2.0, 3.0, 4.0], device='cuda')
out = torch.empty_like(x)
debug_kernel[(1,)](x, out, 4, BLOCK_SIZE=4)
print(f"输出: {out.tolist()}")
print("(tl.device_print 的输出会在 stderr 中)")

## 24.3 常见错误与解决

### 错误1: BLOCK_SIZE 不是 2 的幂
```python
tl.arange(0, 100)  # ✗ 100 不是 2 的幂
tl.arange(0, 128)  # ✓ 128 = 2^7
```

### 错误2: tl.dot 的形状要求
```python
# tl.dot(a, b) 要求:
# - a 的形状 (M, K), b 的形状 (K, N)
# - M, N, K 必须是 16 的倍数（Tensor Core 要求）
# - 输入必须是 float16 或 bfloat16
tl.dot(a_fp32, b_fp32)   # ✗ float32 不行
tl.dot(a_fp16, b_fp16)   # ✓
```

### 错误3: 忘记 mask 导致越界
```python
# 当 n 不是 BLOCK_SIZE 的倍数时
tl.load(ptr + offsets)                      # ✗ 可能越界
tl.load(ptr + offsets, mask=offsets < n)     # ✓ 有 mask 保护
```

### 错误4: constexpr 参数传了运行时值
```python
n = x.shape[0]  # 运行时值
kernel[grid](x, BLOCK=n)  # ✗ 如果 n 每次不同，每次都重新编译
kernel[grid](x, BLOCK=1024)  # ✓ 固定的编译时常量
```

### 错误5: 原子操作没初始化输出
```python
out = torch.empty(1, device='cuda')   # ✗ 未初始化！
tl.atomic_add(out_ptr, val)            # 结果不可预测

out = torch.zeros(1, device='cuda')   # ✓ 初始化为 0
tl.atomic_add(out_ptr, val)            # 正确
```

## 24.4 性能分析工具

### 方法1: triton.testing.do_bench
```python
ms = triton.testing.do_bench(lambda: kernel[grid](args...), warmup=25, rep=100)
```

### 方法2: PyTorch Profiler
```python
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CUDA],
    record_shapes=True,
) as prof:
    kernel[grid](args...)
    torch.cuda.synchronize()

print(prof.key_averages().table())
```

### 方法3: NVIDIA Nsight Systems
```bash
nsys profile python my_triton_script.py
# 生成 .nsys-rep 文件，用 Nsight Systems UI 打开
```

### 方法4: NVIDIA Nsight Compute (NCU)
```bash
ncu --set full python my_triton_script.py
# 详细的 kernel 级别性能分析
# 包括: 寄存器使用, 共享内存, 占用率, 内存带宽利用率
```

In [ ]:
# 实用的性能分析辅助函数
def analyze_gemm_performance(M, N, K, ms, dtype=torch.float16):
    """
    分析 GEMM 的性能指标
    """
    bytes_per_element = 2 if dtype == torch.float16 else 4
    
    # 计算量
    flops = 2 * M * N * K
    tflops = flops / ms * 1e-9
    
    # 内存访问量 (最优情况)
    mem_bytes = (M * K + K * N + M * N) * bytes_per_element
    bandwidth = mem_bytes / ms * 1e-6  # GB/s
    
    # 算术强度
    arithmetic_intensity = flops / mem_bytes  # FLOP/Byte
    
    print(f"╔══════════════════════════════════════════╗")
    print(f"║  GEMM 性能分析: ({M}×{K}) @ ({K}×{N})")
    print(f"╠══════════════════════════════════════════╣")
    print(f"║  执行时间:      {ms:.3f} ms")
    print(f"║  计算吞吐:      {tflops:.1f} TFLOPS")
    print(f"║  最小内存访问:   {mem_bytes/1e6:.1f} MB")
    print(f"║  内存带宽:      {bandwidth:.0f} GB/s")
    print(f"║  算术强度:      {arithmetic_intensity:.1f} FLOP/Byte")
    print(f"╚══════════════════════════════════════════╝")
    
    return tflops, bandwidth, arithmetic_intensity

# 示例
analyze_gemm_performance(2048, 2048, 1024, 0.5)

## 24.5 Roofline 模型

Roofline 模型帮助判断 kernel 是计算瓶颈还是内存瓶颈：

```
TFLOPS
  ↑
  │        ╱ 峰值计算 (如 RTX 3090: 71 TFLOPS FP16)
  │       ╱─────────────────────────────
  │      ╱
  │     ╱  ← 内存带宽限制区域
  │    ╱     (算术强度低的 kernel 在这里)
  │   ╱
  │  ╱    ★ GEMM (高算术强度) → 计算限制
  │ ╱  ★ Softmax (低算术强度) → 内存限制
  │╱
  └─────────────────────────→ 算术强度 (FLOP/Byte)
         拐点 = 峰值TFLOPS / 峰值带宽
```

**关键指标**:
- 算术强度 < 拐点 → 内存瓶颈 → 优化内存访问
- 算术强度 > 拐点 → 计算瓶颈 → 优化计算效率

## 总结

| 调试方法 | 使用场景 |
|----------|----------|
| `TRITON_INTERPRET=1` | 逻辑调试、不需要 GPU |
| `tl.device_print` | GPU 上打印少量调试信息 |
| `do_bench` | 快速性能测量 |
| PyTorch Profiler | 端到端时间分析 |
| Nsight Systems | 时间线可视化 |
| Nsight Compute | 深度 kernel 分析 |

## 练习

1. 用 `TRITON_INTERPRET=1` 调试一个有 bug 的 kernel
2. 用 `do_bench` 测量不同 BLOCK_SIZE 对向量加法的影响
3. 计算你的 GPU 的 roofline 拐点（查找峰值 TFLOPS 和带宽）
4. 分析 Softmax 和 GEMM 各自是什么瓶颈